# Privacy in Machine Learning

## Overview
ML models trained on sensitive data (medical records, financial data, personal communications) can unintentionally memorize and leak private information. This notebook covers privacy attacks, differential privacy as a mathematical defense, and practical tools.

---

## 1. Privacy Attacks on ML Models

### 1.1 Membership Inference Attack
**Goal**: Determine if a specific data point was in the training set.

**Method** (Shokri et al., 2017):
- Models tend to be more confident on training data (overfitting signal)
- Train a meta-classifier on (model output, member/non-member) pairs
- If model gives high confidence on $x$, it was likely in training set

$$P(\text{member} \mid f(x)) > 0.5 \implies \text{infer membership}$$

**Risk**: Reveals who is in a medical trial, financial record, etc.

### 1.2 Model Inversion Attack
**Goal**: Reconstruct training data from model outputs.

$$x^* = \arg\max_x f_c(x) - \lambda R(x)$$

Where $f_c(x)$ is the confidence for class $c$ and $R(x)$ is a regularizer.
Demonstrated on face recognition: reconstructed faces of training subjects.

### 1.3 Data Extraction from LLMs
- LLMs memorize training data verbatim (especially repeated text)
- Prompting with a prefix can cause the model to complete with memorized training data
- Carlini et al. (2021) extracted PII, code, and copyrighted text from GPT-2

---

## 2. Differential Privacy (DP)

Differential privacy provides a **mathematical guarantee** that individual records cannot be identified from model outputs.

### 2.1 Definition
A randomized mechanism $\mathcal{M}: \mathcal{D} \to \mathcal{R}$ is $(\epsilon, \delta)$-differentially private if for all adjacent datasets $D, D'$ (differing in one record) and all subsets $S \subseteq \mathcal{R}$:

$$P[\mathcal{M}(D) \in S] \leq e^\epsilon \cdot P[\mathcal{M}(D') \in S] + \delta$$

- **$\epsilon$ (epsilon)**: privacy budget smaller = more private. Typical values: 0.1 to 10
- **$\delta$ (delta)**: probability of catastrophic failure. Typically $10^{-5}$ to $10^{-8}$
- **Pure DP**: $\delta = 0$ ($(\epsilon, 0)$-DP)

### 2.2 Sensitivity
The $L_2$ sensitivity of query $f$ measures how much one record can change the output:
$$\Delta_2 f = \max_{D, D'} ||f(D) - f(D')||_2$$

### 2.3 Laplace Mechanism
For a scalar query $f: \mathcal{D} \to \mathbb{R}$ with sensitivity $\Delta_1 f$:
$$\mathcal{M}(D) = f(D) + \text{Lap}\left(\frac{\Delta_1 f}{\epsilon}\right)$$

Provides $\epsilon$-DP (pure).

### 2.4 Gaussian Mechanism
For vector queries, adds Gaussian noise:
$$\mathcal{M}(D) = f(D) + \mathcal{N}\left(0, \sigma^2 I\right)$$

Where $\sigma \geq \frac{\Delta_2 f \cdot \sqrt{2 \ln(1.25/\delta)}}{\epsilon}$

Provides $(\epsilon, \delta)$-DP.

### 2.5 Composition Theorems
**Sequential composition**: $k$ mechanisms each $(\epsilon_i, \delta_i)$-DP compose to $(\sum \epsilon_i, \sum \delta_i)$-DP.

**Advanced composition** (tighter bound): after $k$ mechanisms:
$$\epsilon_{total} \approx \epsilon \sqrt{2k \ln(1/\delta)} + k\epsilon(e^\epsilon - 1)$$

---

## 3. DP-SGD (Differentially Private SGD)

The standard algorithm for training ML models with differential privacy (Abadi et al., 2016):

**Algorithm**:
1. Compute per-sample gradients: $g_i = \nabla_\theta L(\theta, x_i)$
2. **Clip** each gradient: $\bar{g}_i = g_i / \max\left(1, \frac{||g_i||_2}{C}\right)$
3. **Add noise**: $\tilde{g} = \frac{1}{B}\left(\sum_i \bar{g}_i + \mathcal{N}(0, \sigma^2 C^2 I)\right)$
4. Update: $\theta \leftarrow \theta - \eta \tilde{g}$

Privacy accounting uses the **moments accountant** or **Rényi DP** to track the privacy budget across all steps.

**Privacy-utility tradeoff**: More privacy (smaller $\epsilon$) → more noise → worse accuracy.

---

## 4. Federated Learning for Privacy

Train models without centralizing data:
1. Server sends model to each client
2. Each client trains locally on private data
3. Only **gradients** (not raw data) are sent to server
4. Server aggregates gradients: $w_{t+1} = w_t - \eta \frac{1}{K}\sum_{k=1}^K \nabla_k$

**Limitation**: Gradients can still leak information (gradient inversion attacks). Solution: combine with DP.

---

## 5. GDPR & Privacy Compliance

- **Data minimization**: only collect what's necessary
- **Purpose limitation**: only use data for stated purpose
- **Right to erasure (right to be forgotten)**: ability to remove a person's data
- **Machine unlearning**: remove influence of specific records without full retraining
- **Article 22**: right not to be subject to solely automated decisions

In [1]:
import numpy as np

# ── Laplace Mechanism ────────────────────────────────────────────
def laplace_mechanism(true_value, sensitivity, epsilon):
    """Add Laplace noise for epsilon-DP."""
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale)
    return true_value + noise

# Example: differentially private count query
true_count = 1000  # number of patients with condition X
sensitivity = 1    # adding/removing one person changes count by 1

print("Laplace Mechanism Differentially Private Count:")
print(f"True count: {true_count}")
for epsilon in [0.1, 0.5, 1.0, 5.0, 10.0]:
    dp_count = laplace_mechanism(true_count, sensitivity, epsilon)
    print(f"  ε={epsilon:<4}: {dp_count:.1f}  (error: {abs(dp_count-true_count):.1f})")

Laplace Mechanism Differentially Private Count:
True count: 1000
  ε=0.1 : 1016.2  (error: 16.2)
  ε=0.5 : 1001.1  (error: 1.1)
  ε=1.0 : 998.8  (error: 1.2)
  ε=5.0 : 1000.0  (error: 0.0)
  ε=10.0: 1000.1  (error: 0.1)


In [2]:
# ── Gaussian Mechanism ──────────────────────────────────────────
def gaussian_mechanism(true_value, l2_sensitivity, epsilon, delta):
    """Add Gaussian noise for (epsilon, delta)-DP."""
    sigma = l2_sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = np.random.normal(0, sigma, size=np.array(true_value).shape)
    return np.array(true_value) + noise, sigma

# DP mean query on vector
true_mean = np.array([0.5, -0.3, 0.8])
dp_mean, sigma = gaussian_mechanism(true_mean, l2_sensitivity=1.0, epsilon=1.0, delta=1e-5)
print(f"True mean:   {true_mean}")
print(f"Private mean: {dp_mean.round(3)}")
print(f"Noise sigma: {sigma:.3f}")

True mean:   [ 0.5 -0.3  0.8]
Private mean: [ 0.089 -3.821 -2.071]
Noise sigma: 4.845


In [3]:
# ── DP-SGD with Opacus ──────────────────────────────────────────
# pip install opacus
# import torch
# import torch.nn as nn
# from opacus import PrivacyEngine
# from opacus.validators import ModuleValidator

# model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# # Attach privacy engine
# privacy_engine = PrivacyEngine()
# model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
#     module=model,
#     optimizer=optimizer,
#     data_loader=train_loader,
#     epochs=10,
#     target_epsilon=5.0,   # privacy budget
#     target_delta=1e-5,
#     max_grad_norm=1.0,    # gradient clipping threshold C
# )

# # Training loop same as normal PyTorch!
# for X_batch, y_batch in train_loader:
#     loss = criterion(model(X_batch), y_batch)
#     optimizer.zero_grad(); loss.backward(); optimizer.step()

# epsilon = privacy_engine.get_epsilon(delta=1e-5)
# print(f"Training complete. (ε={epsilon:.2f}, δ=1e-5)")
print("Opacus DP-SGD template ready same training loop as normal PyTorch")

Opacus DP-SGD template ready same training loop as normal PyTorch


In [4]:
# ── Simple Membership Inference Attack Demo ─────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# Train a model (overfitting on purpose to demonstrate vulnerability)
X_train, y_train = make_classification(n_samples=500, n_features=20, random_state=42)
X_test, y_test = make_classification(n_samples=500, n_features=20, random_state=99)

target_model = LogisticRegression(max_iter=1000, C=100)  # high C = less regularization
target_model.fit(X_train, y_train)

# Confidence-based membership inference
def membership_inference_attack(model, X_member, X_nonmember, threshold=0.7):
    """Simple confidence-based membership inference."""
    proba_member = model.predict_proba(X_member).max(axis=1)
    proba_nonmember = model.predict_proba(X_nonmember).max(axis=1)
    
    tp = (proba_member >= threshold).sum()
    fp = (proba_nonmember >= threshold).sum()
    tn = (proba_nonmember < threshold).sum()
    fn = (proba_member < threshold).sum()
    
    attack_accuracy = (tp + tn) / (tp + tn + fp + fn)
    return attack_accuracy, proba_member.mean(), proba_nonmember.mean()

acc, avg_conf_train, avg_conf_test = membership_inference_attack(
    target_model, X_train, X_test
)
print(f"Avg confidence on training data: {avg_conf_train:.3f}")
print(f"Avg confidence on test data:     {avg_conf_test:.3f}")
print(f"Attack accuracy: {acc:.3f} (0.5 = random guess)")

Avg confidence on training data: 0.882
Avg confidence on test data:     0.809
Attack accuracy: 0.564 (0.5 = random guess)


## Additional Learning Resources

### Foundational Papers
- [The Algorithmic Foundations of Differential Privacy (Dwork & Roth)](https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf) Free book
- [Deep Learning with Differential Privacy DP-SGD (Abadi et al.)](https://arxiv.org/abs/1607.00133)
- [Membership Inference Attacks (Shokri et al.)](https://arxiv.org/abs/1610.05820)
- [Extracting Training Data from LLMs (Carlini et al.)](https://arxiv.org/abs/2012.07805)
- [Federated Learning (McMahan et al.)](https://arxiv.org/abs/1602.05629)

### Libraries
- [Opacus (PyTorch DP)](https://opacus.ai/)
- [TensorFlow Privacy](https://github.com/tensorflow/privacy)
- [PySyft (Federated Learning)](https://github.com/OpenMined/PySyft)
- [Diffprivlib (IBM)](https://diffprivlib.readthedocs.io/)

### Courses
- [Harvard CS208 Applied Privacy for Data Science](https://privacytools.seas.harvard.edu/courses/cs208)